# ПЗ 7 — Распознавание объектов через OpenRouter (nvidia/nemotron-nano-12b-v2-vl)

Отправляем кадры на FastAPI-сервер (Timeweb, европейский IP), который через OpenRouter вызывает мультимодальную модель `nvidia/nemotron-nano-12b-v2-vl:free` — бесплатно, без региональных ограничений.

In [ ]:
!pip install requests pillow tqdm -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

os.makedirs(RESULTS_DIR, exist_ok=True)

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
print(f'кадров всего: {len(frames)}')


In [ ]:
import requests

# FastAPI-сервер на Timeweb с европейским IP
# Внутри сервер обращается к OpenRouter → nvidia/nemotron-nano-12b-v2-vl:free
API_URL = 'http://92.51.37.40:8000'

resp = requests.get(f'{API_URL}/')
print('статус сервера:', resp.json())


In [ ]:
import json
import time
from tqdm.notebook import tqdm

def describe_frame(image_path):
    with open(image_path, 'rb') as f:
        resp = requests.post(
            f'{API_URL}/describe',
            files={'file': (os.path.basename(image_path), f, 'image/jpeg')}
        )
    resp.raise_for_status()
    return resp.json()

# каждый N-й кадр, не больше 20 штук
STEP  = max(1, len(frames) // 20)
BATCH = frames[::STEP][:20]
PAUSE = 3  # пауза между запросами (сек)

print(f'модель: nvidia/nemotron-nano-12b-v2-vl:free (через OpenRouter)')
print(f'обрабатываем {len(BATCH)} кадров с паузой {PAUSE} сек')


In [ ]:
results = []

for i, fname in enumerate(tqdm(BATCH, desc='openrouter')):
    try:
        res = describe_frame(f'{FRAMES_DIR}/{fname}')
        res['frame'] = fname
        results.append(res)
        print(f'{fname}: {res.get("objects", "")}')
    except Exception as e:
        print(f'{fname}: ошибка — {e}')
        results.append({'frame': fname, 'objects': 'ошибка', 'error': str(e)})

    if i < len(BATCH) - 1:
        time.sleep(PAUSE)

with open(f'{RESULTS_DIR}/llm_detections.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f'\nобработано: {len(results)} кадров')
print(f'сохранено: {RESULTS_DIR}/llm_detections.json')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import textwrap

# показываем обработанные кадры с описаниями
cols = 4
rows = (len(results) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5))
axes = axes.flatten()

for i, item in enumerate(results):
    img_path = f'{FRAMES_DIR}/{item["frame"]}'
    try:
        img = mpimg.imread(img_path)
        axes[i].imshow(img)
    except:
        axes[i].set_facecolor('#eee')

    axes[i].axis('off')
    caption = item.get('objects', 'ошибка')
    wrapped = '\n'.join(textwrap.wrap(caption, width=40))
    axes[i].set_title(f'{item["frame"]}\n{wrapped}', fontsize=7, loc='left')

# скрываем пустые ячейки
for j in range(len(results), len(axes)):
    axes[j].axis('off')

plt.suptitle('ПЗ 7 — Кадры с LLM-описаниями (OpenRouter)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/llm_summary.png', dpi=80, bbox_inches='tight')
plt.show()
print('график сохранён: llm_summary.png')
